# 01 · Comprensión de los datos
**Proyecto:** clasificación de enfermedades foliares del cafeto con CNN.
**BRACOL** (`bracol_yolo.zip`, anotaciones YOLO) = dataset principal → se clasifican **recortes de cada caja**.
**JMuBEN / JMuBEN2** = validación externa (hojas completas, etiqueta por imagen).

**Guía:** secciones 15, 16.1 y 16.2 (fuente, data card, calidad de datos).

Este notebook: (1) prepara Drive y verifica `src/common.py`, (2) diagnostica BRACOL y genera los recortes, (3) elimina casi-duplicados entre splits, (4) extrae y muestrea JMuBEN, (5) construye el manifest final, mide la calidad de los datos y deja lista la data card.
**Todo lo que se muestre sale de los archivos; nada se escribe a mano.**

### Antes de empezar
1. En `MyDrive/proyecto_cafe_cnn/data/raw/` deben estar: `bracol_yolo.zip`, `jmuben.zip`, `jmuben2.zip`.
2. `src/common.py` ya debe existir en `MyDrive/proyecto_cafe_cnn/src/` (no se modifica aquí).
3. Activa GPU: *Entorno de ejecución → Cambiar tipo → T4 GPU*.

## 0 · Entorno

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/proyecto_cafe_cnn'
for d in ['src', 'data/raw', 'data/processed', 'models', 'results', 'figures']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

assert os.path.exists(f'{PROJECT}/src/common.py'), 'Falta src/common.py en Drive'
for z in ['bracol_yolo.zip', 'jmuben.zip', 'jmuben2.zip']:
    assert os.path.exists(f'{PROJECT}/data/raw/{z}'), f'Falta data/raw/{z}'
print('Estructura y archivos de entrada OK')

In [ ]:
os.environ['PROJECT_DIR'] = PROJECT
sys.path.insert(0, f'{PROJECT}/src')
import common as C
import numpy as np, pandas as pd, matplotlib.pyplot as plt
P = C.paths()
SEED = 42
C.set_seed(SEED)
print(P)

In [ ]:
# Versiones del entorno (sección Reproducibilidad)
import platform, tensorflow as tf, sklearn, PIL
print('Python', platform.python_version(), '| TF', tf.__version__, '| sklearn', sklearn.__version__, '| PIL', PIL.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
!pip freeze > {PROJECT}/requirements_freeze.txt

## 1 · BRACOL: estructura y diagnóstico
`bracol_yolo.zip` trae dos versiones: **REVISADO** (la que usamos) y **ORIGINAL** (solo para comparar). Las etiquetas son cajas YOLO (`clase cx cy w h`), no una etiqueta por imagen.

In [ ]:
import zipfile, re, glob, collections, shutil
from PIL import Image

DEST = '/content/bracol_yolo'
if not os.path.exists(DEST):
    with zipfile.ZipFile(f"{P['data/raw']}/bracol_yolo.zip") as z:
        print('primer archivo dañado (None = todo bien):', z.testzip())
        z.extractall(DEST)

print('--- data.yaml ---')
for y in glob.glob(f"{DEST}/**/data.yaml", recursive=True):
    print(y); print(open(y).read()[:500])

In [ ]:
NAMES = ['Cercospora', 'Miner', 'Phoma', 'Rust']
IMG = ('.jpg', '.jpeg', '.png', '.bmp')
ROOT = f"{DEST}/BRACOL_REVIEWED_ANNOTATIONS/BRACOL_REVIEWED"
ROOT_ORIG = f"{DEST}/BRACOL-ORIGINAL-ANNOTATIONS/BRACOL-ORIGINAL-DETECT"

def diagnostico(root, tag):
    print(f"\n===== {tag} =====")
    bases = {}; sizes = collections.Counter(); box_side = []
    for split in ['train', 'valid', 'test']:
        img_dir, lab_dir = f"{root}/{split}/images", f"{root}/{split}/labels"
        imgs = sorted(f for f in os.listdir(img_dir) if f.lower().endswith(IMG))
        bases[split] = [f.split('.rf.')[0] for f in imgs]
        cnt = collections.Counter(); n_empty = 0
        for f in imgs:
            stem = os.path.splitext(f)[0]
            lf = f"{lab_dir}/{stem}.txt"
            lines = [l.split() for l in open(lf).read().strip().splitlines() if l.strip()] if os.path.exists(lf) else []
            n_empty += (len(lines) == 0)
            with Image.open(f"{img_dir}/{f}") as im:
                W, H = im.size
            sizes[(W, H)] += 1
            for l in lines:
                cnt[NAMES[int(l[0])]] += 1
                box_side.append(min(float(l[3]) * W, float(l[4]) * H))
        print(f"{split:6s}: {len(imgs):4d} imágenes ({len(set(bases[split]))} únicas por nombre base), {n_empty} sin cajas | cajas: {dict(cnt)}")
    print('tamaños de imagen (ancho, alto):', sizes.most_common(5))
    print('lado menor de la caja (px), percentiles 5/25/50/75/95:', np.percentile(box_side, [5, 25, 50, 75, 95]).round(0))
    for a, b in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]:
        print(f"nombres base compartidos {a} ∩ {b}:", len(set(bases[a]) & set(bases[b])))

diagnostico(ROOT, 'REVISADO (el que usamos)')
diagnostico(ROOT_ORIG, 'ORIGINAL (solo comparación)')

**Lectura del diagnóstico.** Las imágenes miden 2048×1024 y la caja mediana ronda 55 px: si se redimensiona la imagen completa a 224×224, cada lesión queda de unos pocos píxeles. Por eso se clasifican **recortes de cada caja** (con margen de contexto), respetando los splits oficiales (no comparten nombres base). Las imágenes sin cajas no se usan: no sabemos si son hojas sanas o simplemente no se anotaron.

## 2 · Recortes de BRACOL (una muestra por caja)

In [ ]:
OUT = '/content/bracol_crops'                       # local: más rápido que Drive
CROPS_ZIP = f"{P['data/processed']}/bracol_crops.zip"
PAD, MIN_SIDE = 0.25, 48                            # margen de contexto y lado mínimo (px)

if os.path.exists(CROPS_ZIP) and not os.path.exists(OUT):
    with zipfile.ZipFile(CROPS_ZIP) as z:
        z.extractall(OUT)
    print('Recortes restaurados desde Drive')
elif not os.path.exists(OUT):
    n = 0
    for split in ['train', 'valid', 'test']:
        img_dir, lab_dir = f"{ROOT}/{split}/images", f"{ROOT}/{split}/labels"
        for f in sorted(os.listdir(img_dir)):
            if not f.lower().endswith(IMG): continue
            stem = os.path.splitext(f)[0]
            base = stem.split('.rf.')[0]
            lf = f"{lab_dir}/{stem}.txt"
            if not os.path.exists(lf): continue
            lines = [l.split() for l in open(lf).read().strip().splitlines() if l.strip()]
            if not lines: continue
            with Image.open(f"{img_dir}/{f}") as im:
                im = im.convert('RGB'); W, H = im.size
                for k, l in enumerate(lines):
                    cls = NAMES[int(l[0])]
                    cx, cy, w, h = (float(v) for v in l[1:5])
                    cx, cy, w, h = cx * W, cy * H, w * W, h * H
                    side = max(max(w, h) * (1 + 2 * PAD), MIN_SIDE)
                    box = (int(max(0, cx - side / 2)), int(max(0, cy - side / 2)),
                           int(min(W, cx + side / 2)), int(min(H, cy + side / 2)))
                    d = f"{OUT}/{split}/{cls.lower()}"
                    os.makedirs(d, exist_ok=True)
                    im.crop(box).save(f"{d}/{base}__{k}.jpg", quality=95)
                    n += 1
    print('recortes generados:', n)
    shutil.make_archive(CROPS_ZIP[:-4], 'zip', OUT)  # copia en Drive para las próximas sesiones
    print('guardado:', CROPS_ZIP)
else:
    print('Recortes ya presentes en', OUT)
    if not os.path.exists(CROPS_ZIP):               # asegurar la copia en Drive
        shutil.make_archive(CROPS_ZIP[:-4], 'zip', OUT)
        print('guardado:', CROPS_ZIP)

In [ ]:
# Manifest de BRACOL con el mismo código del proyecto (md5, dhash, corruptas...)
bra_all = C.build_manifest(OUT, 'bracol')           # label_raw = carpeta de clase
bra_all['split'] = bra_all.path.str.split('/').str[-3]   # split oficial: train/valid/test
bra_all['label'] = bra_all['label_raw']
bra_all['group'] = bra_all.path.apply(lambda p: os.path.basename(p).split('__')[0])  # imagen de origen
bra_all.to_csv(f"{P['data/processed']}/manifest_bracol_crops.csv", index=False)
print(bra_all.groupby(['split', 'label']).size().unstack(fill_value=0))
print('corruptas:', bra_all.corrupt.sum(), '| md5 duplicados:', bra_all.md5.duplicated().sum())

## 3 · Calidad de los recortes y casi-duplicados entre splits
Se mide el tamaño de los recortes y se buscan casi-duplicados (dHash, distancia de Hamming ≤ 4). Si un grupo cruza splits, se conserva el recorte en la partición de **mayor prioridad (test > val > train)**, para que test quede intacto.

In [ ]:
ok = bra_all[~bra_all.corrupt].copy().reset_index(drop=True)

ok['lado'] = ok[['width', 'height']].max(axis=1)
print(ok.groupby('label').lado.describe()[['count', '25%', '50%', '75%']].round(0))
print('recortes < 64 px (se ampliarán >3.5x):', (ok.lado < 64).sum())

ok['near_grp'] = C.near_dup_groups(ok.dhash.tolist(), max_dist=4)
cruce = ok.groupby('near_grp').split.nunique()
print('grupos casi-duplicados que cruzan splits:', (cruce > 1).sum(), 'de', len(cruce))

In [ ]:
prio = {'test': 0, 'valid': 1, 'train': 2}
ok['prio'] = ok.split.map(prio)
mejor = ok.groupby('near_grp').prio.transform('min')
N_ELIM = int((ok.prio > mejor).sum())
print('recortes eliminados por casi-duplicado entre splits:', N_ELIM)
bra = ok[ok.prio == mejor].copy()
bra['group'] = bra.near_grp.astype(str)             # el grupo ahora es el de casi-duplicados
bra['split'] = bra.split.replace({'valid': 'val'})  # common.py usa 'val'
assert C.check_leakage(bra, cols=('md5', 'group')), 'Hay fuga entre splits'
print('BRACOL tras depurar:', len(bra), 'recortes')

## 4 · JMuBEN: extracción y muestra para validación externa
JMuBEN tiene ~58 mil imágenes repartidas en dos depósitos (`jmuben`: Cercospora, Leaf rust, Phoma; `jmuben2`: Healthy, Miner). Se extrae en disco **local**, se toma una muestra estratificada (500 por clase) y solo esa muestra se copia a Drive.

In [ ]:
SAMPLE = f"{P['data/processed']}/jmuben_sample"
TOTALS = f"{P['results']}/jmuben_totales.csv"
MAP = {'cerscospora': 'cercospora', 'leaf rust': 'rust', 'miner': 'miner',
       'phoma': 'phoma', 'healthy': 'healthy'}

if len(C.list_images(SAMPLE)) >= 2500 and os.path.exists(TOTALS):
    print('Muestra ya en Drive: se reutiliza')
    jm = C.build_manifest(SAMPLE, 'jmuben', seed=SEED)          # etiqueta = carpeta de clase
    jm['label'] = jm.label_raw
else:
    OUTER, JM = '/content/jmuben_outer', '/content/jmuben'
    for d in (OUTER, JM): os.makedirs(d, exist_ok=True)
    # zips externos
    for z in ['jmuben.zip', 'jmuben2.zip']:
        with zipfile.ZipFile(f"{P['data/raw']}/{z}") as f:
            f.extractall(f"{OUTER}/{z[:-4]}")
        print('extraído', z)
    # zips internos (uno por clase): la carpeta toma el nombre de la clase
    for zp in sorted(glob.glob(f"{OUTER}/**/*.zip", recursive=True)):
        clase = re.split(r'-20\d{6}T', os.path.basename(zp))[0].strip()
        with zipfile.ZipFile(zp) as f:
            f.extractall(f"{JM}/{clase}")
    # totales reales por clase (para la data card)
    tot = {MAP[d.lower()]: len(C.list_images(f"{JM}/{d}")) for d in sorted(os.listdir(JM))}
    pd.Series(tot, name='imagenes').to_csv(TOTALS)
    print(tot, '| total:', sum(tot.values()))
    # muestra estratificada
    first_dir = lambda p: os.path.relpath(p, JM).split(os.sep)[0]
    jm = C.build_manifest(JM, 'jmuben', label_fn=first_dir, max_per_label=500, seed=SEED)
    jm['label'] = jm.label_raw.map(MAP)
    assert jm.label.notna().all(), jm.label_raw[jm.label.isna()].unique()
    print('JMuBEN corruptas en la muestra:', jm.corrupt.sum())
    # copiar la muestra a Drive
    newp = []
    for p, lab, h in zip(jm.path, jm.label, jm.md5):
        d = f"{SAMPLE}/{lab}"; os.makedirs(d, exist_ok=True)
        q = f"{d}/{h[:8]}_{os.path.basename(p)}"
        shutil.copy2(p, q); newp.append(q)
    jm['path'] = newp

jm['split'] = 'external'
jm['group'] = jm.md5
print(jm[~jm.corrupt].label.value_counts())

## 5 · Manifest final y fuga entre datasets
Se compara por md5 las **imágenes completas** de BRACOL con las de JMuBEN (los recortes nunca coinciden por md5).

In [ ]:
cols = ['path', 'source', 'label', 'split', 'group', 'md5', 'dhash', 'width', 'height']
final = pd.concat([bra[cols], jm[~jm.corrupt][cols]], ignore_index=True)
final.to_csv(f"{P['data/processed']}/manifest_final.csv", index=False)

full = sum((C.list_images(f"{ROOT}/{s}/images") for s in ['train', 'valid', 'test']), [])
b = {C.file_md5(p) for p in full}
print('md5 compartidos BRACOL (imágenes completas) ∩ JMuBEN:', len(b & set(jm.md5)))
print(final.groupby(['split', 'label']).size().unstack(fill_value=0))

## 6 · Calidad de los datos (guía 16.2)
Completitud, unicidad, validez y actualidad. Se usan los manifests **completos** (antes de depurar) para contar corruptas y duplicados.

In [ ]:
allm = pd.concat([bra_all, jm], ignore_index=True)
q = allm.groupby('source').agg(
    muestras=('path', 'size'),
    corruptas=('corrupt', 'sum'),
    duplicados_exactos=('md5', lambda s: s.dropna().duplicated().sum()),
    ancho_min=('width', 'min'), ancho_med=('width', 'median'), ancho_max=('width', 'max'),
    alto_min=('height', 'min'), alto_med=('height', 'median'), alto_max=('height', 'max'))
q['casi_duplicados_eliminados'] = [N_ELIM if s == 'bracol' else 0 for s in q.index]
q.to_csv(f"{P['results']}/calidad_datos.csv")
display(q)

bal = final.groupby(['source', 'label']).size().unstack(fill_value=0)
bal.to_csv(f"{P['results']}/balance_clases.csv")
display(bal)
display(pd.read_csv(TOTALS, index_col=0).T)   # totales reales de JMuBEN

In [ ]:
# Figura: balance de clases por fuente y split (BRACOL)
t = final[final.source == 'bracol'].groupby(['split', 'label']).size().unstack(fill_value=0).loc[['train', 'val', 'test']]
t.plot(kind='bar', figsize=(7, 4), rot=0)
plt.title('BRACOL: recortes por split y clase'); plt.ylabel('recortes'); plt.tight_layout()
plt.savefig(f"{P['figures']}/balance_bracol.png", dpi=150); plt.show()

## Data card (guía 16.1)
Los campos marcados **VERIFICAR** no salen de los archivos: cópialos de la página original. *No completes campos sin haberlos verificado en la fuente.*

| Campo | BRACOL | JMuBEN |
|---|---|---|
| Fuente / DOI | Mendeley Data 10.17632/yy2k5y8mxg.1; versión YOLO en Roboflow: `jonatan-fragoso/bracol-original-detect` (v1) y `detection-disease-coffe-leafs/bracol-validado` (v3) | Mendeley Data, dos depósitos (`jmuben` y `jmuben2`). **VERIFICAR** DOI de cada uno |
| Propietarios / autores | **VERIFICAR** (página de Mendeley y paper) | **VERIFICAR** |
| Fecha de acceso | **VERIFICAR** (fecha de tu descarga) | **VERIFICAR** |
| País / condiciones de captura | **VERIFICAR** (página del dataset y su paper) | **VERIFICAR** |
| Nº de imágenes | 1744 imágenes anotadas (set revisado: 1213 / 351 / 180); unidad de análisis = recortes de caja (cifras exactas en `balance_clases.csv`) | total real en `jmuben_totales.csv`; muestra de 500 por clase |
| Clases | Cercospora, Miner, Phoma, Rust (4) | Cercospora, Healthy, Leaf rust (= Rust), Miner, Phoma (5) |
| Formato y resolución | Imágenes de 2048×1024; recortes de tamaño variable (`calidad_datos.csv`) | ver `calidad_datos.csv` |
| Duplicados / corruptas | ver `calidad_datos.csv` (incluye casi-duplicados eliminados entre splits) | ver `calidad_datos.csv` |
| Licencia / permiso de uso | CC BY 4.0 según `data.yaml` de Roboflow. **VERIFICAR** también en Mendeley | **VERIFICAR** en las páginas de Mendeley |
| Preprocesamiento previo (Roboflow) | **VERIFICAR** (resize / aumento en las páginas de Roboflow) | **VERIFICAR** |
| Sesgos y limitaciones | Desbalance fuerte (Rust domina; Cercospora y Miner con pocos ejemplos), lesiones pequeñas (Rust mediana ~92 px frente a ~170–290 px en Cercospora y Miner), etiquetas dependientes de cajas anotadas por personas. Contrastar con el paper | Hojas completas vs. recortes de BRACOL (cambio de dominio); muestra de 500 por clase sobre un conjunto mucho mayor; contrastar con el paper (región de captura, aumento previo) |